In [4]:
!pip install decord einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 99.6 MB/s eta 0:00:00:00:0100:01


In [ ]:
!pip install pillow transformers sentencepiece opencv-python  kagglehub

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [ ]:
import os
import kagglehub

# Set đường dẫn bạn muốn lưu (ví dụ: lưu vào ổ Drive của bạn trên Colab)
os.environ["KAGGLEHUB_CACHE"] = "/workspace/project"

# Tiến hành tải

# path = kagglehub.dataset_download("trnphmngcminh/cider-60-38-alpha-0-7")
path = kagglehub.dataset_download("huytduc/msrvtt-10000-video")


print("File đã được tải và lưu tại:", path)

In [ ]:
import kagglehub

os.environ["KAGGLEHUB_CACHE"] = "/workspace/project"

# Download latest version
path = kagglehub.dataset_download("trnphmngcminh/mm-projector")

print("Path to dataset files:", path)

In [7]:
import argparse
import math
import os
from pathlib import Path
from types import SimpleNamespace
from typing import List

import decord
import einops
import torch
import torch.nn as nn
from timm.layers import LayerNorm2d
from timm.models.regnet import RegStage
from transformers import SiglipImageProcessor, SiglipVisionModel


def build_mlp(depth: int, hidden_size: int, output_hidden_size: int) -> nn.Sequential:
    modules = [nn.Linear(hidden_size, output_hidden_size)]
    for _ in range(1, depth):
        modules.append(nn.GELU())
        modules.append(nn.Linear(output_hidden_size, output_hidden_size))
    return nn.Sequential(*modules)


class STCConnector(nn.Module):
    def __init__(self, config, downsample=(2, 2, 2), depth=4, mlp_depth=2):
        super().__init__()
        self.encoder_hidden_size = config.mm_hidden_size  # 1152
        self.hidden_size = config.hidden_size             # 3584
        self.output_hidden_size = config.hidden_size      # 3584
        self.depth = depth
        self.mlp_depth = mlp_depth
        self.downsample = downsample

        if depth != 0:
            self.s1 = RegStage(
                depth=depth,
                in_chs=self.encoder_hidden_size,
                out_chs=self.hidden_size,
                stride=1,
                dilation=1,
                act_layer=nn.SiLU,
                norm_layer=LayerNorm2d,
            )
        else:
            self.s1 = nn.Identity()

        self.sampler = nn.Sequential(
            nn.Conv3d(
                in_channels=self.hidden_size,
                out_channels=self.hidden_size,
                kernel_size=downsample,
                stride=downsample,
                padding=1,
                bias=True,
            ),
            nn.SiLU(),
        )

        if depth != 0:
            self.s2 = RegStage(
                depth=depth,
                in_chs=self.hidden_size,
                out_chs=self.hidden_size,
                stride=1,
                dilation=1,
                act_layer=nn.SiLU,
                norm_layer=LayerNorm2d,
            )
        else:
            self.s2 = nn.Identity()

        self.readout = build_mlp(mlp_depth, self.hidden_size, self.output_hidden_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x:
          [B, T, H, W, D] or [B, T, L, D]
        return:
          [B, L_after, 3584]
        """
        t = x.size(1)

        if x.ndim == 4:
            hw = int(math.sqrt(x.size(2)))
            if hw * hw != x.size(2):
                raise ValueError(f"L={x.size(2)} is not a square number")
            x = einops.rearrange(x, "b t (h w) d -> b d t h w", h=hw, w=hw)
        elif x.ndim == 5:
            x = einops.rearrange(x, "b t h w d -> b d t h w")
        else:
            raise ValueError(f"Unexpected input shape: {x.shape}")

        x = einops.rearrange(x, "b d t h w -> (b t) d h w")
        x = self.s1(x)
        x = einops.rearrange(x, "(b t) d h w -> b d t h w", t=t)

        x = self.sampler(x)
        new_t = x.size(2)

        x = einops.rearrange(x, "b d t h w -> (b t) d h w")
        x = self.s2(x)
        x = einops.rearrange(x, "(b t) d h w -> b (t h w) d", t=new_t)
        x = self.readout(x)
        return x


class STCConnectorV35(STCConnector):
    def __init__(self, config, downsample=(2, 2, 2), depth=4, mlp_depth=2):
        super().__init__(config=config, downsample=downsample, depth=depth, mlp_depth=mlp_depth)
        self.sampler = nn.Sequential(
            nn.Conv3d(
                in_channels=self.hidden_size,
                out_channels=self.hidden_size,
                kernel_size=downsample,
                stride=downsample,
                padding=0,
                bias=True,
            ),
            nn.SiLU(),
        )


def load_stc_weights(stc: nn.Module, mm_projector_path: str) -> None:
    if os.path.isdir(mm_projector_path):
        mm_projector_path = os.path.join(mm_projector_path, "mm_projector.bin")

    state = torch.load(mm_projector_path, map_location="cpu")

    cleaned = {}
    for k, v in state.items():
        nk = k
        for prefix in ["mm_projector.", "model.mm_projector."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v

    missing, unexpected = stc.load_state_dict(cleaned, strict=False)
    print("Loaded STC weights")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)


class SiglipSTC3584Extractor(nn.Module):
    def __init__(
        self,
        siglip_name: str,
        mm_projector_path: str,
        device: str = "cuda",
        dtype: torch.dtype = torch.float32,
    ):
        super().__init__()
        self.device = device
        self.dtype = dtype
        self.siglip_name = siglip_name

        self.processor = SiglipImageProcessor.from_pretrained(siglip_name)
        self.vision = SiglipVisionModel.from_pretrained(siglip_name, torch_dtype=dtype)

        cfg = SimpleNamespace(mm_hidden_size=1152, hidden_size=3584)
        self.stc = STCConnectorV35(cfg)
        load_stc_weights(self.stc, mm_projector_path)

        self.vision.to(device=device, dtype=dtype).eval()
        self.stc.to(device=device, dtype=dtype).eval()

        for p in self.vision.parameters():
            p.requires_grad = False
        for p in self.stc.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def extract_patch_tokens(self, video_path: str, num_frames: int = 16) -> torch.Tensor:
        frames = sample_video_frames(video_path, num_frames=num_frames)
        proc = self.processor(images=frames, return_tensors="pt")
        pixel_values = proc["pixel_values"].to(self.device, dtype=self.dtype)  # [T,3,384,384]

        outputs = self.vision(pixel_values=pixel_values, output_hidden_states=True)
        tokens = outputs.hidden_states[-2]  # [T, L, 1152]

        if tokens.shape[1] == 730:
            tokens = tokens[:, 1:, :]
        elif tokens.shape[1] != 729:
            raise ValueError(f"Unexpected token length {tokens.shape[1]}. Expected 729 or 730.")

        t, l, d = tokens.shape
        hw = int(math.sqrt(l))
        tokens = tokens.view(t, hw, hw, d)  # [T,27,27,1152]
        tokens = tokens.unsqueeze(0)        # [1,T,27,27,1152]
        return tokens

    @torch.no_grad()
    def forward(self, video_path: str, num_frames: int = 16) -> torch.Tensor:
        x = self.extract_patch_tokens(video_path, num_frames=num_frames)  # [1,16,27,27,1152]
        x = self.stc(x)                                                    # [1,1352,3584]
        return x


# -----------------------------
# Video utils
# -----------------------------
def uniform_indices(total_frames: int, num_frames: int) -> List[int]:
    if total_frames <= 0:
        raise ValueError("Video has no frames")
    if total_frames >= num_frames:
        return torch.linspace(0, total_frames - 1, steps=num_frames).long().tolist()
    return list(range(total_frames)) + [total_frames - 1] * (num_frames - total_frames)


def sample_video_frames(video_path: str, num_frames: int = 16) -> List:
    vr = decord.VideoReader(video_path)
    indices = uniform_indices(len(vr), num_frames)
    frames = vr.get_batch(indices).asnumpy()  # [T,H,W,C]
    return [frame for frame in frames]


# -----------------------------
# Save / process
# -----------------------------
def save_feature_tensor(feature: torch.Tensor, output_path: str) -> None:
    feature = feature.squeeze(0).float().cpu().contiguous()  # [1352,3584] FP32
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    torch.save(feature, output_path)


def process_single_video(
    extractor: SiglipSTC3584Extractor,
    video_path: str,
    output_path: str,
    num_frames: int = 16,
) -> None:
    print(f"\n[TEST] Processing single video: {video_path}")
    feat = extractor(video_path, num_frames=num_frames)
    save_feature_tensor(feat, output_path)
    print(f"Saved feature -> {output_path}")
    print(f"Feature shape: {tuple(feat.squeeze(0).shape)}")
    print(f"Feature dtype (saved): torch.float32")


def process_video_folder(
    extractor: SiglipSTC3584Extractor,
    input_dir: str,
    output_dir: str,
    num_frames: int = 16,
    overwrite: bool = False,
) -> None:
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    video_paths = sorted(input_dir.rglob("*.mp4"))
    if not video_paths:
        raise FileNotFoundError(f"No .mp4 videos found in: {input_dir}")

    print(f"\n[BULK] Found {len(video_paths)} videos")
    print(f"Input folder : {input_dir}")
    print(f"Output folder: {output_dir}")

    ok = 0
    skipped = 0
    failed = 0

    for idx, video_path in enumerate(video_paths, start=1):
        rel = video_path.relative_to(input_dir)
        out_path = output_dir / rel.with_suffix(".pt")
        out_path.parent.mkdir(parents=True, exist_ok=True)

        if out_path.exists() and not overwrite:
            skipped += 1
            if idx % 100 == 0 or idx == len(video_paths):
                print(f"[{idx}/{len(video_paths)}] skipped={skipped}, ok={ok}, failed={failed}")
            continue

        try:
            feat = extractor(str(video_path), num_frames=num_frames)
            save_feature_tensor(feat, str(out_path))
            ok += 1
        except Exception as e:
            failed += 1
            print(f"[ERROR] {video_path}: {e}")

        if idx % 50 == 0 or idx == len(video_paths):
            print(f"[{idx}/{len(video_paths)}] ok={ok}, skipped={skipped}, failed={failed}")

    print("\nDone")
    print(f"Success: {ok}")
    print(f"Skipped: {skipped}")
    print(f"Failed : {failed}")


# -----------------------------
# CLI
# -----------------------------
def parse_args():
    parser = argparse.ArgumentParser(description="Offline export features: SigLIP -> STC -> [1352,3584]")

    parser.add_argument("--siglip_name", type=str, default="google/siglip-so400m-patch14-384")
    parser.add_argument("--mm_projector_path", type=str, required=True,
                        help="Path to VideoLLaMA2.1 mm_projector.bin or its folder")
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--num_frames", type=int, default=16)

    parser.add_argument("--test_video", type=str, default=None,
                        help="Path to one .mp4 video for test run")
    parser.add_argument("--test_output", type=str, default="./test_feature.pt",
                        help="Output .pt path for test video")

    parser.add_argument("--input_dir", type=str, default=None,
                        help="Folder containing .mp4 videos")
    parser.add_argument("--output_dir", type=str, default=None,
                        help="Folder to save .pt features")
    parser.add_argument("--overwrite", action="store_true")

    return parser.parse_args()


def main():
    args = parse_args()

    extractor = SiglipSTC3584Extractor(
        siglip_name=args.siglip_name,
        mm_projector_path=args.mm_projector_path,
        device=args.device,
        dtype=torch.float32,
    )

    if args.test_video is not None:
        process_single_video(
            extractor=extractor,
            video_path=args.test_video,
            output_path=args.test_output,
            num_frames=args.num_frames,
        )

    if args.input_dir is not None and args.output_dir is not None:
        process_video_folder(
            extractor=extractor,
            input_dir=args.input_dir,
            output_dir=args.output_dir,
            num_frames=args.num_frames,
            overwrite=args.overwrite,
        )

    if args.test_video is None and not (args.input_dir and args.output_dir):
        print("Nothing to do. Use --test_video for one-video test, or --input_dir + --output_dir for bulk export.")


In [ ]:
# import kagglehub

# path = kagglehub.dataset_download("huytduc/msrvtt-10000-video")

# print(path)

Mounting files to /kaggle/input/datasets/huytduc/msrvtt-10000-video...
/kaggle/input/datasets/huytduc/msrvtt-10000-video


In [9]:
MM_PROJECTOR_PATH = "/kaggle/input/datasets/trnphmngcminh/mm-projector/mm_projector.bin"   # hoặc folder chứa mm_projector.bin
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_FRAMES = 16

In [10]:
extractor = SiglipSTC3584Extractor(
    siglip_name="google/siglip-so400m-patch14-384",
    mm_projector_path=MM_PROJECTOR_PATH,
    device=DEVICE,
    dtype=torch.float32,   # giữ FP32 như bạn đang muốn
)

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/448 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip-so400m-patch14-384
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...26}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...26}.mlp.fc1.weight  

Loaded STC weights
Missing keys: []
Unexpected keys: []


In [13]:
TEST_VIDEO = "/kaggle/input/datasets/huytduc/msrvtt-10000-video/raw_videos/video0.mp4"
TEST_OUTPUT = "/kaggle/working/test_feature.pt"

process_single_video(
    extractor=extractor,
    video_path=TEST_VIDEO,
    output_path=TEST_OUTPUT,
    num_frames=NUM_FRAMES,
)


[TEST] Processing single video: /kaggle/input/datasets/huytduc/msrvtt-10000-video/raw_videos/video0.mp4
Saved feature -> /kaggle/working/test_feature.pt
Feature shape: (1352, 3584)
Feature dtype (saved): torch.float32


In [14]:
import torch

feat = torch.load(TEST_OUTPUT, map_location="cpu")
print(type(feat))
print(feat.shape)
print(feat.dtype)

<class 'torch.Tensor'>
torch.Size([1352, 3584])
torch.float32


In [19]:
%mkdir /kaggle/working/out_put_1
# INPUT_DIR = "/kaggle/input/datasets/huytduc/msrvtt-10000-video/raw_videos"
# OUTPUT_DIR = "/kaggle/working/out_put"

In [18]:
# process_video_folder(
#     extractor=extractor,
#     input_dir=INPUT_DIR,
#     output_dir=OUTPUT_DIR,
#     num_frames=NUM_FRAMES,
#     overwrite=False,
# )

In [ ]:
import os
import pickle
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/datasets/huytduc/msrvtt-10000-video/raw_videos")
OUTPUT_DIR = Path("/kaggle/working/out_put_1")

video_paths = sorted(INPUT_DIR.rglob("*.mp4"))
print("total videos:", len(video_paths))

ok, skipped, failed = 0, 0, 0

for idx, video_path in enumerate(video_paths, start=1):
    rel = video_path.relative_to(INPUT_DIR)
    out_path = OUTPUT_DIR / rel.with_suffix(".pickle")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        skipped += 1
        continue

    try:
        feat = extractor(str(video_path), num_frames=16)   # [1,1352,3584]
        feat = feat.squeeze(0).float().cpu().contiguous().numpy()

        with open(out_path, "wb") as f:
            pickle.dump(feat, f, protocol=pickle.HIGHEST_PROTOCOL)

        ok += 1
    except Exception as e:
        failed += 1
        print(f"[ERROR] {video_path}: {e}")

    if idx % 50 == 0 or idx == len(video_paths):
        print(f"[{idx}/{len(video_paths)}] ok={ok}, skipped={skipped}, failed={failed}")

print("done")
print("ok =", ok)
print("skipped =", skipped)
print("failed =", failed)

total videos: 10000
[50/10000] ok=50, skipped=0, failed=0
[100/10000] ok=100, skipped=0, failed=0


In [2]:
# import os
# import json
# from types import SimpleNamespace

# import torch
# import torch.nn as nn
# import einops
# from huggingface_hub import hf_hub_download
# from timm.models.layers import LayerNorm2d
# from timm.models.regnet import RegStage

# MODEL_ID = "DAMO-NLP-SG/VideoLLaMA2.1-7B-16F-Base"


# def build_mlp(depth, hidden_size, output_hidden_size):
#     modules = [nn.Linear(hidden_size, output_hidden_size)]
#     for _ in range(1, depth):
#         modules.append(nn.GELU())
#         modules.append(nn.Linear(output_hidden_size, output_hidden_size))
#     return nn.Sequential(*modules)


# class STCConnector(nn.Module):
#     def __init__(self, config, downsample=(2, 2, 2), depth=4, mlp_depth=2):
#         super().__init__()
#         self.encoder_hidden_size = config.mm_hidden_size
#         self.hidden_size = config.hidden_size
#         self.output_hidden_size = config.hidden_size

#         if depth != 0:
#             self.s1 = RegStage(
#                 depth=depth,
#                 in_chs=self.encoder_hidden_size,
#                 out_chs=self.hidden_size,
#                 stride=1,
#                 dilation=1,
#                 act_layer=nn.SiLU,
#                 norm_layer=LayerNorm2d,
#             )
#         else:
#             self.s1 = nn.Identity()

#         self.sampler = nn.Sequential(
#             nn.Conv3d(
#                 in_channels=self.hidden_size,
#                 out_channels=self.hidden_size,
#                 kernel_size=downsample,
#                 stride=downsample,
#                 padding=1,
#                 bias=True,
#             ),
#             nn.SiLU(),
#         )

#         if depth != 0:
#             self.s2 = RegStage(
#                 depth=depth,
#                 in_chs=self.hidden_size,
#                 out_chs=self.hidden_size,
#                 stride=1,
#                 dilation=1,
#                 act_layer=nn.SiLU,
#                 norm_layer=LayerNorm2d,
#             )
#         else:
#             self.s2 = nn.Identity()

#         self.readout = build_mlp(mlp_depth, self.hidden_size, self.output_hidden_size)

#     def forward(self, x):
#         t = x.size(1)

#         if x.ndim == 4:
#             hw = int(x.size(2) ** 0.5)
#             assert hw * hw == x.size(2), "L must be a square number"
#             x = einops.rearrange(x, "b t (h w) d -> b d t h w", h=hw, w=hw)
#         elif x.ndim == 5:
#             x = einops.rearrange(x, "b t h w d -> b d t h w")
#         else:
#             raise ValueError(f"Expected 4D or 5D input, got {x.ndim}D")

#         x = einops.rearrange(x, "b d t h w -> (b t) d h w")
#         x = self.s1(x)
#         x = einops.rearrange(x, "(b t) d h w -> b d t h w", t=t)

#         x = self.sampler(x)
#         new_t = x.size(2)

#         x = einops.rearrange(x, "b d t h w -> (b t) d h w")
#         x = self.s2(x)

#         x = einops.rearrange(x, "(b t) d h w -> b (t h w) d", t=new_t)
#         x = self.readout(x)
#         return x


# class STCConnectorV35(STCConnector):
#     def __init__(self, config, downsample=(2, 2, 2), depth=4, mlp_depth=2):
#         super().__init__(config, downsample=downsample, depth=depth, mlp_depth=mlp_depth)
#         self.sampler = nn.Sequential(
#             nn.Conv3d(
#                 in_channels=self.hidden_size,
#                 out_channels=self.hidden_size,
#                 kernel_size=downsample,
#                 stride=downsample,
#                 padding=0,  # v35
#                 bias=True,
#             ),
#             nn.SiLU(),
#         )


# def load_config_dict(model_id_or_dir):
#     if os.path.isdir(model_id_or_dir):
#         path = os.path.join(model_id_or_dir, "config.json")
#     else:
#         path = hf_hub_download(model_id_or_dir, "config.json")

#     with open(path, "r", encoding="utf-8") as f:
#         return json.load(f)


# def unwrap_checkpoint(obj):
#     # xử lý các checkpoint bọc dict nhiều tầng
#     candidate_keys = ["state_dict", "model", "module", "model_state_dict"]
#     while isinstance(obj, dict):
#         found = False
#         for k in candidate_keys:
#             if k in obj and isinstance(obj[k], dict):
#                 obj = obj[k]
#                 found = True
#                 break
#         if not found:
#             break
#     return obj


# def normalize_mm_projector_state_dict(raw_sd, model_keys):
#     """
#     Trả về state_dict đúng format cho riêng module STC.
#     Ưu tiên:
#     1) key local s1... / sampler... / s2... / readout...
#     2) mm_projector.xxx
#     3) model.mm_projector.xxx
#     4) module.mm_projector.xxx
#     """
#     raw_sd = unwrap_checkpoint(raw_sd)

#     if not isinstance(raw_sd, dict):
#         raise TypeError(f"Checkpoint after unwrap is not a dict, got {type(raw_sd)}")

#     raw_keys = list(raw_sd.keys())
#     model_key_set = set(model_keys)

#     # Case 1: file đã là local keys của projector
#     direct_match = {k: v for k, v in raw_sd.items() if k in model_key_set}
#     if len(direct_match) > 0:
#         return direct_match, "direct"

#     # Case 2..n: có prefix
#     prefixes = [
#         "mm_projector.",
#         "model.mm_projector.",
#         "module.mm_projector.",
#     ]

#     for prefix in prefixes:
#         stripped = {}
#         for k, v in raw_sd.items():
#             if k.startswith(prefix):
#                 nk = k[len(prefix):]
#                 if nk in model_key_set:
#                     stripped[nk] = v
#         if len(stripped) > 0:
#             return stripped, prefix

#     # fallback: in key mẫu để debug
#     sample = raw_keys[:20]
#     raise RuntimeError(
#         "Không tìm được key phù hợp cho STC.\n"
#         f"Sample keys trong checkpoint: {sample}\n"
#         f"Số key model cần nạp: {len(model_keys)}"
#     )


# def load_stc_only(model_id_or_dir=MODEL_ID, device="cpu", dtype=torch.float32):
#     cfg = load_config_dict(model_id_or_dir)
#     cfg_ns = SimpleNamespace(**cfg)

#     projector_type = cfg["mm_projector_type"]
#     if projector_type != "stc_connector_v35":
#         raise ValueError(f"Checkpoint này dùng {projector_type}, không phải stc_connector_v35")

#     stc = STCConnectorV35(cfg_ns).to(device=device, dtype=dtype).eval()

#     if os.path.isdir(model_id_or_dir):
#         mm_path = os.path.join(model_id_or_dir, "mm_projector.bin")
#     else:
#         mm_path = hf_hub_download(model_id_or_dir, "mm_projector.bin")

#     raw_sd = torch.load(mm_path, map_location="cpu")
#     norm_sd, source_mode = normalize_mm_projector_state_dict(raw_sd, stc.state_dict().keys())

#     # cast dtype nếu cần
#     norm_sd = {k: v.to(dtype=dtype) for k, v in norm_sd.items()}

#     incompatible = stc.load_state_dict(norm_sd, strict=False)

#     print("projector_type:", projector_type)
#     print("loaded_from:", source_mode)
#     print("num_loaded_keys:", len(norm_sd))
#     print("missing_keys:", incompatible.missing_keys[:10], "..." if len(incompatible.missing_keys) > 10 else "")
#     print("unexpected_keys:", incompatible.unexpected_keys[:10], "..." if len(incompatible.unexpected_keys) > 10 else "")

#     # Nên ép kiểm tra chặt hơn:
#     assert len(incompatible.missing_keys) == 0, f"Still missing: {incompatible.missing_keys[:20]}"
#     assert len(incompatible.unexpected_keys) == 0, f"Unexpected: {incompatible.unexpected_keys[:20]}"

#     return stc, cfg

In [4]:
# def _conv3d_out_size(size, kernel, stride, padding, dilation):
#     return ((size + 2 * padding - dilation * (kernel - 1) - 1) // stride) + 1


# def test_stc_io(
#     stc,
#     cfg,
#     batch_size=1,
#     device="cpu",
#     dtype=torch.float32,
#     num_frames=4,
#     hw=4,
#     input_mode="5d",
# ):
#     """
#     Test nhanh input/output của STCConnector.

#     input_mode:
#         - "5d": input shape [B, T, H, W, D]
#         - "4d": input shape [B, T, L, D], với L = H * W
#     """

#     mm_hidden_size = int(cfg["mm_hidden_size"])
#     hidden_size = int(cfg["hidden_size"])

#     stc = stc.to(device=device, dtype=dtype).eval()

#     if input_mode == "5d":
#         x = torch.randn(
#             batch_size,
#             num_frames,
#             hw,
#             hw,
#             mm_hidden_size,
#             device=device,
#             dtype=dtype,
#         )
#     elif input_mode == "4d":
#         x = torch.randn(
#             batch_size,
#             num_frames,
#             hw * hw,
#             mm_hidden_size,
#             device=device,
#             dtype=dtype,
#         )
#     else:
#         raise ValueError("input_mode must be '4d' or '5d'")

#     with torch.no_grad():
#         y = stc(x)

#     conv = stc.sampler[0]

#     kt, kh, kw = conv.kernel_size
#     st, sh, sw = conv.stride
#     pt, ph, pw = conv.padding
#     dt, dh, dw = conv.dilation

#     out_t = _conv3d_out_size(num_frames, kt, st, pt, dt)
#     out_h = _conv3d_out_size(hw, kh, sh, ph, dh)
#     out_w = _conv3d_out_size(hw, kw, sw, pw, dw)

#     expected_tokens = out_t * out_h * out_w
#     expected_shape = (batch_size, expected_tokens, hidden_size)

#     assert tuple(y.shape) == expected_shape, (
#         f"Output shape sai. Expected {expected_shape}, got {tuple(y.shape)}"
#     )

#     assert torch.isfinite(y).all(), "Output có NaN hoặc Inf"

#     return {
#         "status": "success",
#         "input_mode": input_mode,
#         "input_shape": tuple(x.shape),
#         "output_shape": tuple(y.shape),
#         "expected_shape": expected_shape,
#         "output_dtype": str(y.dtype),
#         "output_device": str(y.device),
#         "all_finite": bool(torch.isfinite(y).all().item()),
#     }


# if __name__ == "__main__":
#     stc, cfg = load_stc_only(MODEL_ID, device="cpu", dtype=torch.float32)

#     result = test_stc_io(
#         stc,
#         cfg,
#         batch_size=1,
#         device="cpu",
#         dtype=torch.float32,
#     )

#     print(result)

projector_type: stc_connector_v35
loaded_from: model.mm_projector.
num_loaded_keys: 113
missing_keys: [] 
unexpected_keys: [] 
{'status': 'success', 'input_mode': '5d', 'input_shape': (1, 4, 4, 4, 1152), 'output_shape': (1, 8, 3584), 'expected_shape': (1, 8, 3584), 'output_dtype': 'torch.float32', 'output_device': 'cpu', 'all_finite': True}


In [5]:
# def check_stc_checkpoint_compatibility(stc, cfg):
#     """
#     Check STC module có phù hợp với checkpoint/config hay không.
#     Không dùng random input.
#     Chỉ kiểm tra architecture, weight shape, input dim, output dim.
#     """

#     report = {}

#     model_sd = stc.state_dict()

#     report["projector_type"] = cfg.get("mm_projector_type")
#     report["mm_hidden_size_from_cfg"] = cfg.get("mm_hidden_size")
#     report["hidden_size_from_cfg"] = cfg.get("hidden_size")

#     report["stc_encoder_hidden_size"] = stc.encoder_hidden_size
#     report["stc_hidden_size"] = stc.hidden_size
#     report["stc_output_hidden_size"] = stc.output_hidden_size

#     # Input dim mà STC yêu cầu
#     report["expected_input_shape_5d"] = (
#         "B",
#         "T",
#         "H",
#         "W",
#         stc.encoder_hidden_size,
#     )

#     report["expected_input_shape_4d"] = (
#         "B",
#         "T",
#         "L",
#         stc.encoder_hidden_size,
#     )

#     # Output dim mà STC tạo ra
#     report["expected_output_shape"] = (
#         "B",
#         "num_tokens_after_downsample",
#         stc.output_hidden_size,
#     )

#     # Check config có khớp object không
#     assert stc.encoder_hidden_size == cfg["mm_hidden_size"], (
#         f"Input hidden size mismatch: "
#         f"stc.encoder_hidden_size={stc.encoder_hidden_size}, "
#         f"cfg.mm_hidden_size={cfg['mm_hidden_size']}"
#     )

#     assert stc.output_hidden_size == cfg["hidden_size"], (
#         f"Output hidden size mismatch: "
#         f"stc.output_hidden_size={stc.output_hidden_size}, "
#         f"cfg.hidden_size={cfg['hidden_size']}"
#     )

#     # Check một số layer quan trọng
#     sampler_conv = stc.sampler[0]

#     report["sampler_in_channels"] = sampler_conv.in_channels
#     report["sampler_out_channels"] = sampler_conv.out_channels
#     report["sampler_kernel_size"] = sampler_conv.kernel_size
#     report["sampler_stride"] = sampler_conv.stride
#     report["sampler_padding"] = sampler_conv.padding

#     assert sampler_conv.in_channels == stc.hidden_size
#     assert sampler_conv.out_channels == stc.hidden_size

#     # Check readout output
#     last_linear = None
#     for m in reversed(stc.readout):
#         if isinstance(m, nn.Linear):
#             last_linear = m
#             break

#     assert last_linear is not None, "Không tìm thấy Linear cuối trong readout"
#     assert last_linear.out_features == cfg["hidden_size"], (
#         f"Readout output mismatch: {last_linear.out_features} != {cfg['hidden_size']}"
#     )

#     report["num_stc_parameters"] = sum(p.numel() for p in stc.parameters())
#     report["num_state_dict_keys"] = len(model_sd)
#     report["status"] = "checkpoint_architecture_compatible"

#     return report

In [6]:
# if __name__ == "__main__":
#     stc, cfg = load_stc_only(
#         MODEL_ID,
#         device="cpu",
#         dtype=torch.float32,
#     )

#     report = check_stc_checkpoint_compatibility(stc, cfg)

#     print(json.dumps(report, indent=2, ensure_ascii=False))

projector_type: stc_connector_v35
loaded_from: model.mm_projector.
num_loaded_keys: 113
missing_keys: [] 
unexpected_keys: [] 
{
  "projector_type": "stc_connector_v35",
  "mm_hidden_size_from_cfg": 1152,
  "hidden_size_from_cfg": 3584,
  "stc_encoder_hidden_size": 1152,
  "stc_hidden_size": 3584,
  "stc_output_hidden_size": 3584,
  "expected_input_shape_5d": [
    "B",
    "T",
    "H",
    "W",
    1152
  ],
  "expected_input_shape_4d": [
    "B",
    "T",
    "L",
    1152
  ],
  "expected_output_shape": [
    "B",
    "num_tokens_after_downsample",
    3584
  ],
  "sampler_in_channels": 3584,
  "sampler_out_channels": 3584,
  "sampler_kernel_size": [
    2,
    2,
    2
  ],
  "sampler_stride": [
    2,
    2,
    2
  ],
  "sampler_padding": [
    0,
    0,
    0
  ],
  "num_stc_parameters": 376889248,
  "num_state_dict_keys": 113,
  "status": "checkpoint_architecture_compatible"
}
